# Continous Frugal Flows

In this notebook we demonstrate the ability for Frugal Flows to identify Marginal Causal Effects

In [1]:
import sys
import os
sys.path.append("../") # go to parent dir

import jax
import jax.random as jr
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import numpy as np
from scipy.stats import rankdata
import scipy.stats as ss
import statsmodels.api as sm
import seaborn as sns
from sklearn.model_selection import KFold
from tqdm import tqdm

# from data.create_sim_data import *
import data_processing_and_simulations as dpns
import data_processing_and_simulations.causl_sim_data_generation as causl_py
from data_processing_and_simulations.run_all_simulations import plot_simulation_results
from validationMethods import matchit
from frugal_flows.causal_flows import independent_continuous_marginal_flow, get_independent_quantiles, train_frugal_flow
from frugal_flows.bijections import UnivariateNormalCDF

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import SignatureTranslatedAnonymousPackage
from rpy2.robjects.vectors import StrVector

# Activate automatic conversion of rpy2 objects to pandas objects
ro.conversion.set_conversion(ro.default_converter + pandas2ri.converter)
base = importr('base')
utils = importr('utils')

# Import the R library causl
try:
    causl = importr('causl')
except Exception as e:
    # PATCHED 2026-06-02: ('causl') was a string (not a 1-tuple), so StrVector
    # iterated it into 'c','a','u','s','l' and CRAN-installed five fake packages
    # for a GitHub-only package -> R subprocess hung. causl is installed via
    # install_rpy2_libraries.py (now conda-aware); fail loudly if it is not.
    raise RuntimeError(
        "R package 'causl' not installed. Run "
        "`micromamba run -n <env> python install_rpy2_libraries.py` first "
        "(installs causl from rje42/causl via remotes::install_github)."
    ) from e

jax.config.update("jax_enable_x64", True)

hyperparams_dict = {
    'learning_rate': 5e-3,
    'RQS_knots': 8,
    'flow_layers': 5,
    'nn_width': 50,
    'nn_depth': 4,    
    'max_patience': 100,
    'max_epochs': 10000
}
causal_hyperparams = {
    'RQS_knots': 8,
    'flow_layers': 5,
    'nn_width': 50,
    'nn_depth': 4,   
}

SEED = 0
NUM_ITER = 25
NUM_SAMPLES = 25000
TRUE_PARAMS = {'ate': 1, 'const': 0, 'scale': 1}
CAUSAL_PARAMS = [2, 5]

/Users/danielmanela/micromamba/envs/frugal-flows-flowjax/lib/python3.13/site-packages/zepid/datasets/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
def run_outcome_regression(data):
    Z_cont = data.get('Z_cont', None)
    Z_disc = data.get('Z_disc', None)
    if Z_cont == None:
        Z_full = Z_disc
    elif Z_disc == None:
        Z_full = Z_cont
    else:
        Z_full = jnp.hstack([Z_cont, Z_disc])
    Z_cols = [f"Z{i+1}" for i in range(Z_full.shape[1])]
    df = pd.DataFrame(jnp.hstack([data['Y'], data['X'], Z_full]), columns=['Y', 'X', *Z_cols])
    X_vars = df[['X', *Z_cols]]
    X_vars = sm.add_constant(X_vars)
    Y_var = df['Y']
    model = sm.OLS(Y_var, X_vars).fit()
    coefficient_X = model.params['X']
    coefficient_const = model.params['const']
    return coefficient_X, coefficient_const

In [3]:
def dict_to_dataframe(data):
    # Extract the data from the dictionary
    X = np.array(data['X']).flatten()
    Y = np.array(data['Y']).flatten()
    
    # Initialize a dictionary to construct the DataFrame
    df_dict = {'X': X, 'Y': Y}
    
    # Initialize the Z column index
    z_index = 1
    
    # Process Z_cont
    if 'Z_cont' in data and data['Z_cont'] is not None:
        Z_cont = np.array(data['Z_cont'])
        for i in range(Z_cont.shape[1]):
            df_dict[f'Z{z_index}'] = Z_cont[:, i]
            z_index += 1
    
    # Process Z_disc (assuming similar structure if it were present)
    if 'Z_disc' in data and data['Z_disc'] is not None:
        Z_disc = np.array(data['Z_disc'])
        for i in range(Z_disc.shape[1]):
            df_dict[f'Z{z_index}'] = Z_disc[:, i]
            z_index += 1
    
    # Create the DataFrame
    df = pd.DataFrame(df_dict)
    
    return df

In [4]:
data = causl_py.generate_mixed_samples(10000, CAUSAL_PARAMS, 3)
run_outcome_regression(data)

(np.float64(5.1982946334456415), np.float64(1.0871989366635488))

## Checking for the Causal Effect

### Gaussians

#### Frugal Flow

In [5]:
# gaussian_covariates_results = causl_py.run_simulations(
#     causl_py.generate_gaussian_samples, 
#     seed=SEED, 
#     num_samples=NUM_SAMPLES, 
#     num_iter=NUM_ITER, 
#     causal_params=CAUSAL_PARAMS,
#     hyperparams_dict=hyperparams_dict,
#     causal_model_args={'ate': 0., 'const': 1., 'scale': 1}
# )
# gaussian_covariates_results

In [6]:
# display(gaussian_covariates_results.mean())
# display(gaussian_covariates_results.std())

In [7]:
# plt.figure(figsize=(12, 6))

# # Boxplot
# box = gaussian_covariates_results.boxplot(column=["ate", "const", "scale"], grid=False)

# # Adding lines for the true parameters
# plt.axhline(y=TRUE_PARAMS['ate'], color='r', linestyle='--', label='True ate')
# plt.axhline(y=TRUE_PARAMS['const'], color='g', linestyle='--', label='True const')
# plt.axhline(y=TRUE_PARAMS['scale'], color='b', linestyle='--', label='True scale')

# # Adding title and labels
# plt.title('Box and Whisker Plot for ATE, Const, and Scale')
# plt.ylabel('Values')
# plt.ylim([0.80, 1.20])
# plt.legend()

#### Outcome Regression

In [8]:
# gaussian_coeffs = {'ate': [], 'const': []}
# for i in range(NUM_ITER):
#     data = causl_py.generate_gaussian_samples(N=NUM_SAMPLES, causal_params=CAUSAL_PARAMS, seed=i)
#     coeff_X, coeff_const = run_outcome_regression(data)
#     gaussian_coeffs['ate'].append(coeff_X)
#     gaussian_coeffs['const'].append(coeff_const)
# gaussian_outcome_coeffs = pd.DataFrame.from_dict(gaussian_coeffs)
# gaussian_outcome_coeffs

In [9]:
# print(gaussian_outcome_coeffs.mean())
# print(gaussian_outcome_coeffs.std())

### Mixed Gaussian and Gamma Outcomes

In [10]:
N_test = 20000
Z_disc, Z_cont, X, Y = causl_py.generate_mixed_samples(N_test, CAUSAL_PARAMS, N_test).values()

In [11]:
uz_samples = causl_py.generate_uz_samples(Z_cont=Z_cont, Z_disc=None, use_marginal_flow=False, seed=0, frugal_flow_hyperparams=hyperparams_dict)

In [12]:
uz_samples['uz_samples'].shape

(20000, 4)

In [13]:
frugal_flow, losses = causl_py.train_frugal_flow(
    key=jr.PRNGKey(0),
    y=Y,
    u_z=uz_samples['uz_samples'],
    # u_z = jr.uniform(jr.PRNGKey(1000), shape=(N_test, 4)),
    condition=X,
    **hyperparams_dict,
    causal_model='gaussian',
    causal_model_args={'ate': jnp.array([-7.]), 'const': 3., 'scale': 5}
)

  2%|▏         | 153/10000 [01:58<2:06:41,  1.30it/s, train=1.09, val=1.17 (Max patience reached)]


In [14]:
run_outcome_regression({'X': X, 'Y': Y, 'Z_cont': Z_cont})

(np.float64(5.298538914215113), np.float64(0.9895632786541291))

In [15]:
causal_margin = frugal_flow.bijection.bijections[-1].bijection.bijections[0]

In [16]:
print(causal_margin.ate[0])
print(causal_margin.const)
print(causal_margin.scale)

4.956998529089632
2.0272880782557094
1.02728151958931


In [17]:
continous_covariates_results = causl_py.run_simulations(
    causl_py.generate_mixed_samples, 
    seed=SEED, 
    num_samples=NUM_SAMPLES, 
    num_iter=NUM_ITER, 
    causal_params=CAUSAL_PARAMS,
    hyperparams_dict=hyperparams_dict,
    causal_model_args={'ate': jnp.array([0.]), 'const': 0., 'scale': 1}
)

  1%|▏         | 148/10000 [02:32<2:49:28,  1.03s/it, train=1.1, val=1.15 (Max patience reached)]


[array([4.98589744, 1.98902166, 0.99219936])]


  2%|▏         | 154/10000 [02:36<2:47:09,  1.02s/it, train=1.11, val=1.15 (Max patience reached)]


[array([4.98589744, 1.98902166, 0.99219936]), array([4.96815846, 2.02487283, 1.02029661])]


  2%|▏         | 180/10000 [02:59<2:43:18,  1.00it/s, train=1.11, val=1.17 (Max patience reached)]


...

[array([4.98589744, 1.98902166, 0.99219936]), array([4.96815846, 2.02487283, 1.02029661]), array([5.00643874, 1.9781572 , 1.01463871]), array([4.97727725, 1.9911992 , 0.99674837]), array([5.00742393, 1.98169372, 1.00966117]), array([4.98593374, 2.02514887, 1.00727683]), array([4.96076619, 2.04444719, 0.99720925]), array([4.97502853, 2.05258152, 1.02574473]), array([4.99640736, 1.99378138, 1.0091166 ]), array([4.93656867, 2.06051221, 0.99275963]), array([4.75664806, 2.24203314, 1.02005559]), array([4.90735695, 2.09078804, 1.01474841]), array([4.97919066, 2.04441576, 0.98425703]), array([4.90306683, 2.07709385, 1.01122546]), array([4.99679675, 2.0037563 , 0.99987244]), array([4.98704385, 2.00219953, 0.99349301]), array([5.0372791 , 2.00667109, 1.00617095]), array([5.01534695, 1.959956  , 1.00171608]), array([4.99451038, 1.98642427, 1.00849389]), array([4.85893657, 2.12141906, 1.00723684]), array([4.96299462, 2.04946464, 1.01646135]), array([4.96655798, 2.01178788, 0.99458875]), array([4.

  1%|▏         | 147/10000 [02:16<2:32:04,  1.08it/s, train=1.1, val=1.18 (Max patience reached)]


[array([4.98589744, 1.98902166, 0.99219936]), array([4.96815846, 2.02487283, 1.02029661]), array([5.00643874, 1.9781572 , 1.01463871]), array([4.97727725, 1.9911992 , 0.99674837]), array([5.00742393, 1.98169372, 1.00966117]), array([4.98593374, 2.02514887, 1.00727683]), array([4.96076619, 2.04444719, 0.99720925]), array([4.97502853, 2.05258152, 1.02574473]), array([4.99640736, 1.99378138, 1.0091166 ]), array([4.93656867, 2.06051221, 0.99275963]), array([4.75664806, 2.24203314, 1.02005559]), array([4.90735695, 2.09078804, 1.01474841]), array([4.97919066, 2.04441576, 0.98425703]), array([4.90306683, 2.07709385, 1.01122546]), array([4.99679675, 2.0037563 , 0.99987244]), array([4.98704385, 2.00219953, 0.99349301]), array([5.0372791 , 2.00667109, 1.00617095]), array([5.01534695, 1.959956  , 1.00171608]), array([4.99451038, 1.98642427, 1.00849389]), array([4.85893657, 2.12141906, 1.00723684]), array([4.96299462, 2.04946464, 1.01646135]), array([4.96655798, 2.01178788, 0.99458875]), array([4.

  1%|▏         | 131/10000 [02:03<2:34:56,  1.06it/s, train=1.1, val=1.16 (Max patience reached)]

[array([4.98589744, 1.98902166, 0.99219936]), array([4.96815846, 2.02487283, 1.02029661]), array([5.00643874, 1.9781572 , 1.01463871]), array([4.97727725, 1.9911992 , 0.99674837]), array([5.00742393, 1.98169372, 1.00966117]), array([4.98593374, 2.02514887, 1.00727683]), array([4.96076619, 2.04444719, 0.99720925]), array([4.97502853, 2.05258152, 1.02574473]), array([4.99640736, 1.99378138, 1.0091166 ]), array([4.93656867, 2.06051221, 0.99275963]), array([4.75664806, 2.24203314, 1.02005559]), array([4.90735695, 2.09078804, 1.01474841]), array([4.97919066, 2.04441576, 0.98425703]), array([4.90306683, 2.07709385, 1.01122546]), array([4.99679675, 2.0037563 , 0.99987244]), array([4.98704385, 2.00219953, 0.99349301]), array([5.0372791 , 2.00667109, 1.00617095]), array([5.01534695, 1.959956  , 1.00171608]), array([4.99451038, 1.98642427, 1.00849389]), array([4.85893657, 2.12141906, 1.00723684]), array([4.96299462, 2.04946464, 1.01646135]), array([4.96655798, 2.01178788, 0.99458875]), array([4.

In [18]:
print(continous_covariates_results.mean())
print(continous_covariates_results.std())

ate      4.958733
const    2.038390
scale    1.006505
dtype: float64
ate      0.065835
const    0.066221
scale    0.010635
dtype: float64


In [ ]:
matchit_ate_list = []
for i in tqdm(range(NUM_ITER)):
    data = causl_py.generate_mixed_samples(20000, CAUSAL_PARAMS, i)
    matching_results = matchit(outcome='Y', treatment='X', data=dict_to_dataframe(data))
    matchit_ate_list.append(matching_results[0][0])
print(np.mean(matchit_ate_list))
print(np.std(matchit_ate_list))

  0%|          | 0/25 [00:00<?, ?it/s]

ℹ The package "optmatch" is required.
✖ Would you like to install it?

1: Yes
2: No



In [ ]:
print(np.mean(matchit_ate_list))
print(np.std(matchit_ate_list))

#### Outcome Regression

In [ ]:
outcome_coeffs = {'ate': [], 'const': []}
for i in range(NUM_ITER):
    data = causl_py.generate_mixed_samples(N=NUM_SAMPLES, causal_params=CAUSAL_PARAMS, seed=i)
    coeff_X, coeff_const = run_outcome_regression(data)
    outcome_coeffs['ate'].append(coeff_X)
    outcome_coeffs['const'].append(coeff_const)
outcome_coeffs = pd.DataFrame.from_dict(outcome_coeffs)
outcome_coeffs

In [ ]:
print(outcome_coeffs.mean())
print(outcome_coeffs.std())

In [ ]:
plt.figure(figsize=(12, 6))

# Boxplot
box = continous_covariates_results.boxplot(column=["ate", "const", "scale"], grid=False)

# Adding lines for the true parameters
plt.axhline(y=TRUE_PARAMS['ate'], color='r', linestyle='--', label='True ate')
plt.axhline(y=TRUE_PARAMS['const'], color='g', linestyle='--', label='True const')
plt.axhline(y=TRUE_PARAMS['scale'], color='b', linestyle='--', label='True scale')

# Adding title and labels
plt.title('Box and Whisker Plot for ATE, Const, and Scale')
plt.ylabel('Values')
plt.ylim([0.80, 1.20])
plt.legend()

### Mixed Continuous and Discrete (Small)

In [ ]:
discrete_small_covariates_results = causl_py.run_simulations(
    causl_py.generate_discrete_samples, 
    seed=SEED, 
    num_samples=NUM_SAMPLES, 
    num_iter=NUM_ITER, 
    causal_params=CAUSAL_PARAMS,
    hyperparams_dict=hyperparams_dict,
    causal_model_args={'ate': jnp.array([0.]), 'const': 0., 'scale': 1}
)

In [ ]:
discrete_small_covariates_results

In [ ]:
print(discrete_small_covariates_results.mean())
print(discrete_small_covariates_results.std())

In [ ]:
matchit_ate_list_small_discrete = []
for i in tqdm(range(NUM_ITER)):
    data = causl_py.generate_discrete_samples(NUM_SAMPLES, CAUSAL_PARAMS, i)
    matching_results = matchit(outcome='Y', treatment='X', data=dict_to_dataframe(data))
    matchit_ate_list_small_discrete.append(matching_results[0][0])
print(np.mean(matchit_ate_list_small_discrete))
print(np.std(matchit_ate_list_small_discrete))

#### Outcome Regression

In [ ]:
outcome_coeffs = {'ate': [], 'const': []}
for i in range(NUM_ITER):
    data = causl_py.generate_discrete_samples(N=NUM_SAMPLES, causal_params=CAUSAL_PARAMS, seed=i)
    coeff_X, coeff_const = run_outcome_regression(data)
    outcome_coeffs['ate'].append(coeff_X)
    outcome_coeffs['const'].append(coeff_const)
outcome_coeffs = pd.DataFrame.from_dict(outcome_coeffs)
outcome_coeffs

In [ ]:
print(outcome_coeffs.mean())
print(outcome_coeffs.std())

In [ ]:
plt.figure(figsize=(12, 6))

# Boxplot
box = discrete_small_covariates_results.boxplot(column=["ate", "const", "scale"], grid=False)

# Adding lines for the true parameters
plt.axhline(y=TRUE_PARAMS['ate'], color='r', linestyle='--', label='True ate')
plt.axhline(y=TRUE_PARAMS['const'], color='g', linestyle='--', label='True const')
plt.axhline(y=TRUE_PARAMS['scale'], color='b', linestyle='--', label='True scale')

# Adding title and labels
plt.title('Box and Whisker Plot for ATE, Const, and Scale')
plt.ylabel('Values')
plt.ylim([0.80, 1.20])
plt.legend()

### Mixed Continuous and Discrete (Large)

In [ ]:
hyperparams_dict_large = hyperparams_dict.copy()
# hyperparams_dict_large['learning_rate'] = 1e-3
discrete_big_covariates_results = causl_py.run_simulations(
    causl_py.generate_many_discrete_samples, 
    seed=SEED, 
    num_samples=NUM_SAMPLES, 
    num_iter=NUM_ITER, 
    causal_params=CAUSAL_PARAMS,
    hyperparams_dict=hyperparams_dict_large,
    causal_model_args={'ate': jnp.array([0.]), 'const': 0., 'scale': 1}
)

In [ ]:
discrete_big_covariates_results

In [ ]:
print(discrete_big_covariates_results.mean())
print(discrete_big_covariates_results.std())

#### Outcome Regression

In [ ]:
outcome_coeffs = {'ate': [], 'const': []}
for i in range(NUM_ITER):
    data = causl_py.generate_many_discrete_samples(N=NUM_SAMPLES, causal_params=CAUSAL_PARAMS, seed=i)
    coeff_X, coeff_const = run_outcome_regression(data)
    outcome_coeffs['ate'].append(coeff_X)
    outcome_coeffs['const'].append(coeff_const)
outcome_coeffs = pd.DataFrame.from_dict(outcome_coeffs)
outcome_coeffs

In [ ]:
print(outcome_coeffs.mean())
print(outcome_coeffs.std())

In [ ]:
plt.figure(figsize=(12, 6))

# Boxplot
box = discrete_big_covariates_results.boxplot(column=["ate", "const", "scale"], grid=False)

# Adding lines for the true parameters
plt.axhline(y=TRUE_PARAMS['ate'], color='r', linestyle='--', label='True ate')
plt.axhline(y=TRUE_PARAMS['const'], color='g', linestyle='--', label='True const')
plt.axhline(y=TRUE_PARAMS['scale'], color='b', linestyle='--', label='True scale')

# Adding title and labels
plt.title('Box and Whisker Plot for ATE, Const, and Scale')
plt.ylabel('Values')
plt.ylim([0.80, 1.20])
plt.legend()

In [ ]:
matchit_ate_list_many_discrete = []
for i in tqdm(range(NUM_ITER)):
    data = causl_py.generate_many_discrete_samples(NUM_SAMPLES, CAUSAL_PARAMS, i)
    matching_results = matchit(outcome='Y', treatment='X', data=dict_to_dataframe(data))
    matchit_ate_list_many_discrete.append(matching_results[0][0])
print(np.mean(matchit_ate_list_many_discrete))
print(np.std(matchit_ate_list_many_discrete))

### Mixed Continuous and Discrete (Large and Sparse)

In [ ]:
# discrete_sparse_covariates_results = causl_py.run_simulations(
#     causl_py.generate_many_discrete_samples_sparse, 
#     seed=SEED, 
#     num_samples=NUM_SAMPLES, 
#     num_iter=NUM_ITER, 
#     causal_params=CAUSAL_PARAMS,
#     hyperparams_dict=hyperparams_dict,
#     causal_model_args={'ate': 0., 'const': 0., 'scale': 1}
# )
# discrete_sparse_covariates_results

In [ ]:
# print(discrete_sparse_covariates_results.mean())
# print(discrete_sparse_covariates_results.std())

#### Outcome Regression

In [ ]:
outcome_coeffs = {'ate': [], 'const': []}
for i in range(NUM_ITER):
    data = causl_py.generate_many_discrete_samples(N=NUM_SAMPLES, causal_params=CAUSAL_PARAMS, seed=i)
    coeff_X, coeff_const = run_outcome_regression(data)
    outcome_coeffs['ate'].append(coeff_X)
    outcome_coeffs['const'].append(coeff_const)
outcome_coeffs = pd.DataFrame.from_dict(outcome_coeffs)
outcome_coeffs

In [ ]:
# plt.figure(figsize=(12, 6))

# # Boxplot
# box = discrete_sparse_covariates_results.boxplot(column=["ate", "const", "scale"], grid=False)

# # Adding lines for the true parameters
# plt.axhline(y=TRUE_PARAMS['ate'], color='r', linestyle='--', label='True ate')
# plt.axhline(y=TRUE_PARAMS['const'], color='g', linestyle='--', label='True const')
# plt.axhline(y=TRUE_PARAMS['scale'], color='b', linestyle='--', label='True scale')

# # Adding title and labels
# plt.title('Box and Whisker Plot for ATE, Const, and Scale')
# plt.ylabel('Values')
# plt.ylim([0.80, 1.20])
# plt.legend()